# 22 — Free re-fit before implementation (second Opus review, 2026-09-10)

A second deep review (requested before wiring anything into production
code) found a bug of the **same class** as the original headline finding:
notebooks 15/17/19's calibrated blend `(T_cnn, T_baseline, w)` was
LOFO-fit against a **6-member** CNN ensemble (one checkpoint per seed,
averaged across the 6 variant families), but `submission_src/main.py` is
about to ship a **150-member** average. By this project's own documented
mechanism (averaging more imperfectly-correlated members produces
increasingly under-confident probabilities -- see
`project_dat_parkinson_strategic_roadmap.md`), the CNN-side sharpening
was still climbing at k=6; freezing that calibration would ship an
under-sharpened (too conservative) blend.

This notebook does five free (CPU-only, zero submissions) things the
review asked for, before any constant gets frozen into `src/`:

1. **Pool the honest OOF ceiling** -- all 5 seeds' 6-variant arrays
   averaged into one 30-member array (30 = the largest ensemble any row
   can honestly be OOF for, since each `{prefix}_oof_seed{s}.npy` holds
   exactly one held-out checkpoint's prediction per row).
2. **Replace the 3-axis grid search with `LogisticRegression(fit_intercept=True)`
   on `[logit(cnn_p), logit(baseline_p)]`.** The review showed
   `(T_cnn, T_baseline, w)` is really only 2 degrees of freedom
   (`a=w/T_cnn`, `b=(1-w)/T_baseline`) with no intercept -- this also adds
   the free intercept that can absorb a train/test base-rate mismatch.
3. **Row-wise 5-fold CV** (`evaluate.make_folds`, this project's own
   fold design) instead of seed-wise LOFO -- the seed-wise `sd=0.0058`
   measured seed noise on *identical rows*, not row-generalization error.
4. **Trace the CNN coefficient vs. ensemble size** and extrapolate to
   k=150, to check (not assume) that a k=30 fit is close to saturated.
5. **Two more pre-registered comparisons the review flagged as free**:
   probability-space vs. logit-space checkpoint pooling, and the
   6-variant vs. 7-variant(+denoise) ensemble comparison -- the 25
   `rung4_denoise_*` checkpoints are already trained and on disk,
   excluded from the ensemble only for "different preprocessing", which
   is exactly what makes them the most decorrelated variant available
   (the cheapest possible substitute for roadmap item 6, the never-run
   architecture-diversity check).

Also produces the CNN-only fallback calibration for `src/submission.py`'s
degenerate-baseline-mask path (currently returns the raw, uncalibrated
CNN probability), and checks a base-rate-shrinkage epsilon against the
review's flagged tail risk in the `1e-6` probability clip.

**Data handling**: loads real row-level labels and existing OOF prediction
arrays, so per the AI-assistant data rule (`README.md`) this is
**[RUN ME]** — run it yourself, share back only the printed aggregate
numbers. CPU-only, no GPU, no volume cache — seconds to low tens of
seconds total.

In [ ]:
# [RUN ME] -- loads real row-level labels + existing OOF prediction arrays.
# CPU-only, no GPU, no volume cache -- self-contained, does not assume any
# earlier cell/notebook ran in this kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

import config
import evaluate

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
y_true = labeled_df[config.TARGET_COLUMN].to_numpy()
family = labeled_df["inplane_family"].to_numpy()

repeat_seeds = list(range(config.SEED, config.SEED + 5))
# the 6 currently-adopted variants (plain augment -- flip-TTA was dropped
# in notebook 21; see project_dat_parkinson_strategic_roadmap.md)
CURRENT_VARIANT_PREFIXES = ["rung3", "rung4_familybias", "rung4_lrsched", "rung4_augment",
                             "rung4_classweight", "rung4_fixedepoch"]

# 30 individually-OOF CNN "members" (6 variants x 5 seeds). Each array
# element is exactly one held-out checkpoint's prediction for that row --
# by construction every element is OOF regardless of variant/seed, so
# pooling all 30 stays honest.
cnn_members = {
    (prefix, s): np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in CURRENT_VARIANT_PREFIXES for s in repeat_seeds
}
denoise_members = {
    ("rung4_denoise", s): np.load(config.DATA_PROCESSED / f"rung4_denoise_oof_seed{s}.npy")
    for s in repeat_seeds
}
cnn_arrays = list(cnn_members.values())
denoise_arrays = list(denoise_members.values())

baseline_oof_repeats = [np.load(config.DATA_PROCESSED / f"baseline_oof_seed{s}.npy") for s in repeat_seeds]
baseline_pooled = np.mean(baseline_oof_repeats, axis=0)

print(f"{len(cnn_arrays)} CNN members loaded (6 variants x 5 seeds), "
      f"{len(denoise_arrays)} denoise members, {len(baseline_oof_repeats)} baseline repeats pooled.")

In [ ]:
# [RUN ME] (no new data access -- defines helpers used by every cell below).
EPS = 1e-6


def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def prob_mean(arrays):
    return np.mean(arrays, axis=0)


def logit_mean(arrays):
    return 1.0 / (1.0 + np.exp(-np.mean([to_logit(a) for a in arrays], axis=0)))


FOLDS = evaluate.make_folds(y_true, family, n_splits=config.N_FOLDS, random_state=config.RANDOM_STATE)


def new_unregularized_logreg():
    """Plain (unregularized) logistic regression. sklearn >=1.8 deprecated
    `penalty=None` in favor of `C=np.inf` for the identical fit -- see
    https://scikit-learn.org/stable/whats_new -- use C=np.inf everywhere
    in this notebook to avoid the FutureWarning without changing results.
    """
    return LogisticRegression(C=np.inf, max_iter=1000)


def row_cv_score(X, y=y_true, folds=FOLDS):
    """Honest row-wise CV log loss of a logistic-regression calibrated
    blend fit on feature matrix X -- replaces the seed-wise LOFO grid
    search (notebooks 15/17/19). Reuses this project's own fold design
    (evaluate.make_folds) so a row's calibration params are never fit on
    data that includes that row.
    """
    scores = []
    for train_idx, test_idx in folds:
        clf = new_unregularized_logreg()
        clf.fit(X[train_idx], y[train_idx])
        p_test = clf.predict_proba(X[test_idx])[:, 1]
        scores.append(evaluate.log_loss_score(y[test_idx], p_test))
    return np.array(scores)


def full_fit(X, y=y_true):
    """Full-data fit -- the params to actually ship. The row_cv_score
    above (fit on 4/5 of the rows each time) is the honest number to
    trust for how well params like these will do on unseen rows.
    """
    clf = new_unregularized_logreg()
    clf.fit(X, y)
    return clf.coef_[0], clf.intercept_[0]


def blend_features(cnn_p, baseline_p):
    return np.column_stack([to_logit(cnn_p), to_logit(baseline_p)])


print(f"{len(FOLDS)} row-wise folds built (evaluate.make_folds, stratified on target x family).")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# PRE-REGISTERED COMPARISON A: probability-space vs. logit-space pooling
# of the 30-member CNN ensemble, each scored through the new row-wise-CV
# logistic calibrated blend. One comparison; adopt whichever wins, ties
# toward prob-mean (what production already computes with a running sum --
# no reason to add complexity for a null result).
cnn_prob_mean = prob_mean(cnn_arrays)
cnn_logit_mean = logit_mean(cnn_arrays)

scores_prob = row_cv_score(blend_features(cnn_prob_mean, baseline_pooled))
scores_logit = row_cv_score(blend_features(cnn_logit_mean, baseline_pooled))

print(f"prob-space pooling (k=30):  row-CV log loss = {scores_prob.mean():.4f} (sd={scores_prob.std(ddof=1):.4f})")
print(f"logit-space pooling (k=30): row-CV log loss = {scores_logit.mean():.4f} (sd={scores_logit.std(ddof=1):.4f})")
print(f"delta (logit - prob): {scores_logit.mean() - scores_prob.mean():+.4f}")

USE_LOGIT_POOLING = scores_logit.mean() < scores_prob.mean()
pool_fn = logit_mean if USE_LOGIT_POOLING else prob_mean
cnn_pooled = pool_fn(cnn_arrays)
final_scores = scores_logit if USE_LOGIT_POOLING else scores_prob
print(f"\n-> adopted pooling method: {'logit-space' if USE_LOGIT_POOLING else 'probability-space'}")

print(f"\nhonest row-wise CV calibrated-blend score (k=30, this notebook's method): "
      f"mean={final_scores.mean():.4f} sd={final_scores.std(ddof=1):.4f}")
print("for comparison -- notebook 19's seed-wise LOFO (k=6): mean=0.3740 sd=0.0058")
print("for comparison -- notebook 19's pooled candidate (k=6, same-data optimism): log loss=0.3646")

(a, b), c = full_fit(blend_features(cnn_pooled, baseline_pooled))
print(f"\nfull-data fit (params to ship, at k=30): a={a:.4f}  b={b:.4f}  c={c:.4f}")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Finding-1 diagnostic: trace the CNN-side coefficient `a` vs. ensemble
# size k, using random k-subsets of the 30 available members (R draws per
# k, averaged for a smoother trend), then extrapolate to k=150
# (production's real ensemble size). Diagnostic only -- checks whether
# the k=30 fit adopted above is close to saturated, doesn't gate a
# decision on its own.
rng = np.random.RandomState(config.RANDOM_STATE)
K_VALUES = [1, 2, 3, 5, 6, 10, 15, 20, 25, 30]
R_DRAWS = 20

a_by_k = []
for k in K_VALUES:
    a_draws = []
    for _ in range(R_DRAWS):
        subset_idx = rng.choice(len(cnn_arrays), size=k, replace=False)
        subset = [cnn_arrays[i] for i in subset_idx]
        pooled_k = pool_fn(subset)
        (a_k, _), _ = full_fit(blend_features(pooled_k, baseline_pooled))
        a_draws.append(a_k)
    a_by_k.append(float(np.mean(a_draws)))
    print(f"k={k:>2}: a={np.mean(a_draws):.4f} (sd over {R_DRAWS} draws={np.std(a_draws):.4f})")

inv_k = 1.0 / np.array(K_VALUES, dtype=float)
slope, intercept_a = np.polyfit(inv_k, a_by_k, 1)
a_extrapolated_150 = intercept_a + slope / 150
print(f"\nlinear fit in 1/k: a ~= {intercept_a:.4f} + {slope:.4f}/k")
print(f"extrapolated a at k=150: {a_extrapolated_150:.4f}  (vs. k=30's fitted a={a:.4f})")
print(f"difference: {a_extrapolated_150 - a:+.4f}  "
      "(small -> k=30 fit is a safe stand-in for k=150; large -> re-derive a from the extrapolation)")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# PRE-REGISTERED COMPARISON B: current 6-variant (30-member) ensemble vs.
# 7-variant (+denoise, 35-member) ensemble, using the pooling method
# adopted in the cell above. One comparison; adopt only on a win, per
# this project's own discipline (notebooks 16/18). This is the cheapest
# available substitute for roadmap item 6 (bounded architecture check,
# never run) -- denoise is the only on-disk variant that differs on the
# preprocessing axis rather than the training-knob axis.
cnn_plus_denoise_pooled = pool_fn(cnn_arrays + denoise_arrays)

scores_6variant = final_scores  # already computed above, same method
scores_7variant = row_cv_score(blend_features(cnn_plus_denoise_pooled, baseline_pooled))

print(f"6-variant ensemble (current):  row-CV log loss = {scores_6variant.mean():.4f} (sd={scores_6variant.std(ddof=1):.4f})")
print(f"7-variant ensemble (+denoise): row-CV log loss = {scores_7variant.mean():.4f} (sd={scores_7variant.std(ddof=1):.4f})")
print(f"delta (+denoise - current): {scores_7variant.mean() - scores_6variant.mean():+.4f}")
print("DECISION RULE (mechanical): adopt denoise as a 7th variant only if this delta is negative.")

MECHANICAL_ADOPT_DENOISE = scores_7variant.mean() < scores_6variant.mean()
print(f"mechanical rule says: {'adopt' if MECHANICAL_ADOPT_DENOISE else 'keep 6-variant'} "
      f"(delta={scores_7variant.mean() - scores_6variant.mean():+.4f} vs. sd={scores_7variant.std(ddof=1):.4f} "
      f"-> {abs(scores_7variant.mean() - scores_6variant.mean()) / scores_7variant.std(ddof=1):.1%} of 1 sd)")

# USER OVERRIDE (2026-09-10): the mechanical rule only checks the sign,
# not whether the delta clears the measured noise floor. -0.0016 is ~6%
# of this cell's own sd (0.025) -- well inside noise, same pattern that
# got flip-TTA dropped in notebook 21 despite also winning mechanically.
# Given only 1 real submission remains, the user chose not to add
# denoise's implementation complexity (second preprocessing path in
# main.py, ~4.1x inference cost for those 25 checkpoints) for a delta
# indistinguishable from noise. Keeping the checkpoint-loading code above
# so the comparison stays reproducible, but the composition actually
# shipped is 6-variant.
ADOPT_DENOISE = False
final_cnn_pooled = cnn_plus_denoise_pooled if ADOPT_DENOISE else cnn_pooled
final_composition_scores = scores_7variant if ADOPT_DENOISE else scores_6variant
print(f"\n-> shipped composition: {'7-variant (+denoise)' if ADOPT_DENOISE else '6-variant (current, user override)'}")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# BUGFIX: cell op04's (a, b, c) were fit on cnn_pooled (the 6-variant
# composition, before this cell's denoise decision). If op06 adopted the
# 7-variant composition, those coefficients don't correspond to the
# ensemble actually being shipped (final_cnn_pooled). Re-fit on whichever
# composition op06 adopted so the shipped params match the shipped
# ensemble.
if ADOPT_DENOISE:
    (a_final, b_final), c_final = full_fit(blend_features(final_cnn_pooled, baseline_pooled))
    print(f"composition changed (7-variant adopted) -- re-fit full-data params: "
          f"a={a_final:.4f}  b={b_final:.4f}  c={c_final:.4f}")
    print(f"(vs. op04's stale 6-variant fit: a={a:.4f}  b={b:.4f}  c={c:.4f})")
else:
    a_final, b_final, c_final = a, b, c
    print(f"composition unchanged (6-variant kept) -- op04's (a, b, c) are already correct: "
          f"a={a_final:.4f}  b={b_final:.4f}  c={c_final:.4f}")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# CNN-only fallback calibration for src/submission.py's degenerate-
# baseline-mask path (currently returns the raw, uncalibrated CNN
# probability when extract_baseline_features fails). Fit on whichever CNN
# composition was adopted in the cell above.
X_cnn_only = to_logit(final_cnn_pooled).reshape(-1, 1)
fallback_cv_scores = row_cv_score(X_cnn_only)
(a1,), c1 = full_fit(X_cnn_only)
raw_cnn_score = evaluate.log_loss_score(y_true, final_cnn_pooled)

print(f"CNN-only fallback calibration (params to ship): a1={a1:.4f}  c1={c1:.4f}")
print(f"row-wise CV log loss (calibrated, CNN-only): mean={fallback_cv_scores.mean():.4f} sd={fallback_cv_scores.std(ddof=1):.4f}")
print(f"for comparison -- raw uncalibrated CNN pooled log loss: {raw_cnn_score:.4f}")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Optional: row-wise-CV-fit base-rate shrinkage epsilon
# (p' = (1-eps)*p + eps*BASE_RATE, applied after the calibrated blend) --
# a cheap hedge against the current 1e-6 probability clip's tail risk (one
# confidently-wrong row at 1e-6 costs ~0.023 log loss, roughly half this
# whole roadmap's cumulative local-CV gain).
EPS_GRID = np.arange(0.0, 0.051, 0.005)
X_blend = blend_features(final_cnn_pooled, baseline_pooled)

eps_scores = {}
for eps in EPS_GRID:
    fold_scores = []
    for train_idx, test_idx in FOLDS:
        clf = new_unregularized_logreg()
        clf.fit(X_blend[train_idx], y_true[train_idx])
        p_test = clf.predict_proba(X_blend[test_idx])[:, 1]
        p_shrunk = (1 - eps) * p_test + eps * config.BASE_RATE
        fold_scores.append(evaluate.log_loss_score(y_true[test_idx], p_shrunk))
    eps_scores[float(eps)] = float(np.mean(fold_scores))
    print(f"eps={eps:.3f}: row-CV log loss = {eps_scores[float(eps)]:.4f}")

best_eps = min(eps_scores, key=eps_scores.get)
print(f"\nbest eps={best_eps:.3f} (log loss={eps_scores[best_eps]:.4f}) vs. eps=0.000 ({eps_scores[0.0]:.4f})")
print("DECISION RULE: adopt best_eps only if it beats eps=0 by more than the row-CV sd "
      "(cell above's fold sd is a reasonable proxy); otherwise ship eps=0 and rely on "
      "widening the probability clip alone for the tail-risk fix.")

**What we're looking for:** five things, all free (CPU-only, zero
submissions), all requested by the second Opus review before any
constant gets frozen into `src/submission.py`/`submission_src/main.py`:
the right pooling method (prob vs. logit space), an honest row-wise-CV
calibrated-blend recipe (replacing the seed-wise LOFO grid search), a
check that a k=30 fit is close enough to k=150 to ship, whether the
already-trained denoise checkpoints add ensemble value, and calibration
params for the fallback path + tail-risk shrinkage.

**What we found:** *(paste: cell op04's prob-vs-logit delta and the
adopted pooling method + its row-wise CV mean/sd; cell op05's k-trace
table and the k=30-vs-k=150 extrapolated difference; cell op06's
6-variant-vs-7-variant delta and which composition was adopted; the
bugfix cell's a_final/b_final/c_final (re-fit on whichever composition
op06 actually adopted -- op04's own a/b/c are stale once op06 adopts the
7-variant composition); cell op07's fallback a1/c1 and CV score; cell
op08's eps sweep and whether shrinkage was adopted)*

**Decision / next step:** *(the final recipe to ship is: CNN composition
= whichever cell op06 adopted, pooled via cell op04's method, blended
with the classical baseline using the bugfix cell's (a_final, b_final,
c_final) -- NOT op04's raw (a, b, c), which are only correct if op06 kept
the 6-variant composition -- or a re-derived `a` from cell op05's
extrapolation if the k=30-vs-150 gap looks large -- plus cell op07's
fallback params for the degenerate-mask path and cell op08's eps (or just
the widened clip) for the tail-risk hedge. Once this is settled:
implement. Wire into `src/submission.py::combine_predictions` (TDD, new
signature -- takes CNN logit(s), baseline logit, and the fixed (a,b,c)
plus fallback (a1,c1) and eps), update `submission_src/main.py`
(checkpoint list per the adopted composition, GPU-resident batched
inference, shared single volume load for CNN+baseline, parallel
preprocessing, no-sklearn-pickle baseline reconstruction, widened clip,
throttled logging -- see `project_dat_parkinson_strategic_roadmap.md` for
the full implementation-risk checklist from the second Opus review),
extend `scripts/build_submission_assets.py`, rebuild, local Docker smoke
test, platform smoke test, then a real submission. Also resolve the
submission-cap rolling-window question on the platform before deciding
how many submissions are actually available.)*